In [ ]:
import xpress as xp
import pandas as pd
import numpy as np

class TimetablingModel:
    """Full University Timetabling Model: Redistribution + Strict Weeks + Stability + Multi-Objective."""
    
    def __init__(self, students_df, events_df, weeks_df, rooms_df):
        self.students_df = students_df
        self.events_df = events_df
        self.weeks_df = weeks_df
        self.rooms_df = rooms_df
        
        xp.init()
        self.model = xp.problem(name="Timetabling")
        
        self.events, self.weeks, self.rooms = [], [], []
        self.days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
        self.time_slots = [f"{h:02d}:{m:02d}" for h in range(9, 18) for m in [0, 30]]
        
        self.event_size, self.event_duration, self.event_weeks = {}, {}, {}
        self.event_name, self.room_capacity, self.room_campus = {}, {}, {}
        self.room_building, self.curricula = {}, {}
        self.event_original_room, self.event_original_campus = {}, {}
        
        self.x = {} 
        self.delta = {} 
        print("✓ Model initialized")

    def build_data(self, max_events=500, max_rooms=40):
        print("Building data structures...")
        
        teaching_weeks = self.weeks_df[self.weeks_df['Week Type'] == 'Other']['Week Number'].unique()
        if len(teaching_weeks) == 0: teaching_weeks = self.weeks_df['Week Number'].unique()
        self.weeks = sorted([int(w) for w in teaching_weeks if pd.notna(w)])

        def parse_weeks(week_str):
            if pd.isna(week_str): return self.weeks
            w_set = set()
            for part in str(week_str).split(','):
                part = part.strip()
                if '-' in part:
                    s, e = part.split('-')
                    w_set.update(range(int(s), int(e) + 1))
                else:
                    w_set.add(int(part))
            return sorted([w for w in w_set if w in self.weeks])

        self.events = self.events_df['Event ID'].unique()[:max_events].tolist()
        for e in self.events:
            row = self.events_df[self.events_df['Event ID'] == e].iloc[0]
            self.event_name[e] = row.get('Event Name', 'Unknown')
            self.event_size[e] = int(row.get('Event Size', 0)) if pd.notna(row.get('Event Size')) else 0
            dur = row.get('Duration (minutes)', 60)
            self.event_duration[e] = max(1, int(np.ceil(float(dur) / 30)))
            self.event_weeks[e] = parse_weeks(row.get('Weeks'))
            self.event_original_room[e] = row.get('Room')
            self.event_original_campus[e] = row.get('Campus', 'Unknown')

        room_count = 0
        for _, row in self.rooms_df.iterrows():
            if room_count >= max_rooms: break
            if row.get('Campus') == 'Holyrood': continue
            r_id = row['Id']
            self.rooms.append(r_id)
            self.room_capacity[r_id] = int(row.get('Capacity', 0))
            self.room_campus[r_id] = row.get('Campus', 'Central')
            self.room_building[r_id] = row.get('Building_Name', 'Unknown')
            room_count += 1

        student_events = self.students_df.groupby('AnonID')['Event ID'].apply(set).to_dict()
        c_id = 0
        for events in list(student_events.values())[:5000]:
            relevant = [e for e in events if e in self.events]
            if len(relevant) > 1:
                self.curricula[c_id] = relevant
                c_id += 1
        
        try:
            travel_df = pd.read_csv('Travel_Times.csv') 
            self.travel_matrix = {}
            for _, row in travel_df.iterrows():
                c_from, c_to, mins = row['Campus From'], row['Campus To'], row['Travel time (mins)']
                slots = int(np.ceil(mins / 30))
                self.travel_matrix[(c_from, c_to)] = slots
                self.travel_matrix[(c_to, c_from)] = slots
        except:
            self.travel_matrix = {}
        print(f"✓ Data built: {len(self.events)} events, {len(self.rooms)} rooms")

    def build_model(self):
        print("Building High-Performance Multi-Objective Model...")
        
        stability_penalties, wed_policy_penalties = [], []
        campus_penalties, relocation_penalties = [], []
        tier1_penalties, tier2_penalties = [], []
        
        wed_afternoon = [f"{h:02d}:{m:02d}" for h in range(13, 18) for m in [0, 30]]

        # 1. Variable Creation & Basic Penalties
        for e in self.events:
            sz, dur = self.event_size[e], self.event_duration[e]
            orig_c, orig_r = self.event_original_campus.get(e), self.event_original_room.get(e)
            is_lecture = any(word in self.event_name.get(e, "").lower() for word in ["lecture", "whole class"])

            for d in self.days:
                valid_slots = self.time_slots[:len(self.time_slots) - dur + 1]
                for t in valid_slots:
                    for r in self.rooms:
                        cp, camp = self.room_capacity[r], self.room_campus[r]
                        if cp < 0.7 * sz: continue 

                        for w in self.event_weeks.get(e, []):
                            x_var = xp.var(vartype=xp.binary, name=f"x_{e}_{d}_{t}_{r}_{w}")
                            self.x[e,d,t,r,w] = x_var
                            self.model.addVariable(x_var)
                            
                            # TIERED CAPACITY
                            if cp < sz and cp >= 0.9 * sz:
                                v1 = xp.var(lb=0); self.model.addVariable(v1)
                                self.model.addConstraint(v1 >= (sz - cp) * x_var)
                                tier1_penalties.append(50 * v1)
                            elif cp < 0.9 * sz:
                                v2 = xp.var(lb=0); self.model.addVariable(v2)
                                self.model.addConstraint(v2 >= (sz - cp) * x_var)
                                tier2_penalties.append(500 * v2)

                            # WEDNESDAY POLICY
                            if d == 'Wednesday' and t in wed_afternoon and is_lecture:
                                wed_policy_penalties.append(500 * x_var)

                            # CAMPUS PREFERENCE
                            p_c = 10 if camp == 'Lauriston' else (20 if camp == 'New College' else 0)
                            if p_c > 0: campus_penalties.append(p_c * x_var)

                            # RELOCATION (Non-Holyrood)
                            if orig_c != 'Holyrood':
                                if r != orig_r: relocation_penalties.append(25 * x_var)
                                if camp != orig_c: relocation_penalties.append(50 * x_var)

        # 2. HEURISTIC: WEEKLY STABILITY (The Anchor Rule)
        for e in self.events:
            weeks = self.event_weeks.get(e, [])
            if len(weeks) <= 1: continue 
            w_anchor = weeks[0]
            
            for w_other in weeks[1:]:
                d_var = xp.var(vartype=xp.binary, name=f"delta_{e}_{w_other}")
                self.delta[e, w_other] = d_var
                self.model.addVariable(d_var)
                stability_penalties.append(1000 * d_var)
                
                for d in self.days:
                    for t in self.time_slots:
                        for r in self.rooms:
                            if (e,d,t,r,w_anchor) in self.x and (e,d,t,r,w_other) in self.x:
                                self.model.addConstraint(d_var >= self.x[e,d,t,r,w_anchor] - self.x[e,d,t,r,w_other])

        # 3. HARD CONSTRAINTS
        for e in self.events:
            for w in self.event_weeks.get(e, []):
                self.model.addConstraint(xp.Sum(self.x[e,d,t,r,w] for (ev,d,t,r,wk) in self.x.keys() if ev==e and wk==w) == 1)

        for r in self.rooms:
            for w in self.weeks:
                for d in self.days:
                    for t_idx, t in enumerate(self.time_slots):
                        occ = [self.x[e,d,self.time_slots[t_idx-k],r,w] for e in self.events 
                               if w in self.event_weeks.get(e, []) for k in range(self.event_duration[e])
                               if t_idx-k >= 0 and (e,d,self.time_slots[t_idx-k],r,w) in self.x]
                        if occ: self.model.addConstraint(xp.Sum(occ) <= 1)

        for events in self.curricula.values():
            for w in self.weeks:
                for d in self.days:
                    for t_idx, t in enumerate(self.time_slots):
                        clash = [self.x[e,d,self.time_slots[t_idx-k],r,w] for e in events for r in self.rooms 
                                 for k in range(self.event_duration[e])
                                 if t_idx-k >= 0 and (e,d,self.time_slots[t_idx-k],r,w) in self.x]
                        if clash: self.model.addConstraint(xp.Sum(clash) <= 1)

        # 4. OBJECTIVE
        self.model.setObjective(
            xp.Sum(stability_penalties) + xp.Sum(wed_policy_penalties) + 
            xp.Sum(tier1_penalties) + xp.Sum(tier2_penalties) + 
            xp.Sum(campus_penalties) + xp.Sum(relocation_penalties), 
            sense=xp.minimize
        )
        print(f"✓ Model built with {len(self.x)} binary variables.")

    def solve(self, time_limit=300):
        print(f"Solving (Limit: {time_limit}s)...")
        self.model.controls.maxtime = -time_limit
        self.model.solve()
        status = self.model.attributes.solstatus
        if status in [xp.SolStatus.FEASIBLE, xp.SolStatus.OPTIMAL]:
            print(f"Final Objective Value: {self.model.attributes.objval:.2f}")
            return True
        return False

    def extract_solution(self):
        print("🚀 Extracting detailed schedule...")
        results = []
        wed_window = [f"{h:02d}:{m:02d}" for h in range(13, 18) for m in [0, 30]]
        
        for (e_id, d, t, r_id, w), x_var in self.x.items():
            if self.model.getSolution(x_var) > 0.5:
                sz, cp = self.event_size.get(e_id, 0), self.room_capacity.get(r_id, 0)
                alert = "None"
                if sz > cp: alert = "Minor Squeeze" if cp/sz >= 0.9 else "Major Squeeze"
                
                is_lecture = any(word in self.event_name.get(e_id, "").lower() for word in ["lecture", "whole class"])
                wed_violation = "Yes" if (d == 'Wednesday' and t in wed_window and is_lecture) else "No"

                results.append({
                    'Week': w, 'Day': d, 'Time': t, 'Event_ID': e_id,
                    'Event_Name': self.event_name.get(e_id), 'Event_Size': sz,
                    'Room_Capacity': cp, 'Utilization_Pct': round((sz/cp)*100, 1) if cp > 0 else 0,
                    'Crowding_Alert': alert, 'Wednesday_Violation': wed_violation,
                    'Sched_Room': r_id, 'Sched_Campus': self.room_campus.get(r_id),
                    'Orig_Campus': self.event_original_campus.get(e_id),
                    'Was_Relocated': 'Yes' if r_id != self.event_original_room.get(e_id) else 'No'
                })
        
        df = pd.DataFrame(results)
        if not df.empty:
            stability_breaks = sum(1 for v in self.delta.values() if self.model.getSolution(v) > 0.5)
            print("\n" + "="*40 + "\nHEURISTIC PERFORMANCE AUDIT\n" + "="*40)
            print(f"Stability Deviations: {stability_breaks}")
            print(f"Wednesday Violations: {len(df[df['Wednesday_Violation'] == 'Yes'])}")
            print(f"Minor Squeezes:       {len(df[df['Crowding_Alert'] == 'Minor Squeeze'])}")
            print(f"Major Squeezes:       {len(df[df['Crowding_Alert'] == 'Major Squeeze'])}")
            
            day_map = {d: i for i, d in enumerate(self.days)}
            df['d_idx'] = df['Day'].map(day_map)
            df = df.sort_values(['Week', 'd_idx', 'Time']).drop('d_idx', axis=1)
        return df

def run_optimization():
    try:
        e_df, r_df = pd.read_csv('Event_data.csv'), pd.read_csv('Rooms_data.csv')
        s_df, st_df = pd.read_csv('Semester1.csv'), pd.read_csv('Student_data.csv')
    except Exception as err:
        print(f"File Error: {err}"); return

    model = TimetablingModel(st_df, e_df, s_df, r_df)
    model.build_data(max_events=500, max_rooms=40)
    model.build_model()
    
    if model.solve():
        df = model.extract_solution()
        holyrood_events = df[df['Orig_Campus'] == 'Holyrood']
        if not holyrood_events.empty:
            print("\n--- Holyrood Absorption Summary ---")
            print(holyrood_events['Sched_Campus'].value_counts())
            central_pct = (len(holyrood_events[holyrood_events['Sched_Campus'] == 'Central']) / len(holyrood_events)) * 100
            print(f"Success Rate (Central Absorption): {central_pct:.2f}%")
        
        df.to_csv('MultiObj_TT.csv', index=False)
        print(f"✅ Success! Saved to MultiObj_TT.csv")
    else:
        print("❌ No feasible solution found.")

if __name__ == "__main__":
    run_optimization()